In [15]:
import torch

torch.manual_seed(1337)
B, T, C = 4, 8, 2
x = torch.randn(B, T, C)

torch.Size([4, 8, 2])

torch.Size([4, 8, 2])

In [25]:
# naive version: use loops
# goal is x[b, t] = mean_{i<=t} x[b, i]
xbow = torch.zeros(B, T, C) # bag of words
for b in range(B):
    for t in range(T):
        xprev = x[b, :t+1] # (t, C)
        xbow[b, t] = torch.mean(xprev, 0)

In [26]:
# version 2: use matrix operations
weights = torch.tril(torch.ones(T, T))
weights = weights / weights.sum(1, keepdim=True)
xbow2 = weights @ x # (B, T, T) @ (B, T, C) --------> (B, T, C)

#torch.allclose(xbow, xbow2) gives false positive, so we need to relax the tolerance a bit
torch.allclose(xbow, xbow2, atol=1e-6, rtol=1e-4) 

True

In [27]:
import torch.nn.functional as F

# version 3: use softmax
tril = torch.tril(torch.ones(T, T))
weights = torch.zeros((T,T))
weights = weights.masked_fill(tril == 0, float('-inf'))
weights = F.softmax(weights, dim=-1)
xbow3 = weights @ x
torch.allclose(xbow, xbow3, atol=1e-6, rtol=1e-4)

True

In [ ]:
import torch.nn as nn

# veersion 4: self-attention
torch.manual_seed(1337)
B, T, C = 4, 8, 32
x = torch.randn(B, T, C)

# single head performs self-attention
head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)
k = key(x) # (B, T, 16)
q = query(x)  # (B, T, 16)
weights = q @ k.transpose(-2, -1) # (B, T, 16) @ (B, 16, T) ---> (B, T, T)  * head_size**-0.5

tril = torch.tril(torch.ones(T, T))
#weights = torch.zeros((T,T))
weights = weights.masked_fill(tril == 0, float('-inf'))
weights = F.softmax(weights, dim=-1)

v = value(x)
out = weights @ v

out.shape

torch.Size([4, 8, 16])